# 07 · Reading a plate by redrawing it

The YOLO26s reader is kept as it is. After it, the 5 best valid readings of a plate are drawn as clean plates,
fitted to the photo (corners, brightness, blur), and the reading whose drawing fits best is chosen.

**Inputs:** the `bangla-plate-reading` dataset and the YOLO26s reader `best.pt`.

In [ ]:
%pip install -q ultralytics==8.4.46

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

DATA = next(p for p in INPUT.rglob("bangla-plate-reading") if p.is_dir())
WEIGHTS = list(INPUT.rglob("best.pt"))
print(DATA)
print(WEIGHTS)

In [ ]:
classes = json.loads((DATA / "classes.json").read_text(encoding="utf-8"))
bangla = [c["bangla"] for c in classes]
kind = [c["kind"] for c in classes]

test_images = sorted((DATA / "images" / "test").iterdir())
print(len(classes), "classes")
print(len(test_images), "test plates")

## Plate text

Same as notebooks 05 and 06: district, word and letter from left to right, then the six digits as `xx-xxxx`.
The true text of a test plate comes from its label file.

In [ ]:
def plate_text(chars):
    # chars: list of (class id, x centre)
    chars = sorted(chars, key=lambda c: c[1])
    head = " ".join(bangla[c] for c, x in chars if kind[c] != "digit")
    digits = "".join(bangla[c] for c, x in chars if kind[c] == "digit")
    if len(digits) == 6:
        digits = digits[:2] + "-" + digits[2:]
    return " ".join(part for part in [head, digits] if part)


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]


def read_labels(image_path, split="test"):
    # YOLO label file -> list of (class id, x centre, y centre, width, height)
    label = DATA / "labels" / split / (image_path.stem + ".txt")
    boxes = []
    for line in label.read_text().splitlines():
        p = line.split()
        if p:
            boxes.append((int(p[0]), *map(float, p[1:])))
    return boxes


def read_truth(image_path, split="test"):
    return plate_text([(b[0], b[1]) for b in read_labels(image_path, split)])


for path in test_images[:3]:
    print(path.name[:50], "->", read_truth(path))
print(edit_distance("ঢাকা মেট্রো গ ২৪-২৯৫৫", "ঢাকা মেট্রো ২৪-২৯৫৫"))

## Baseline: current reader on the test set

The reader at 0.25 confidence, characters sorted left to right, as in the deployed AI service.
This must match notebook 05: 77.4% on all 939 plates (75.7% bd-lpr, 77.9% blpd).

In [ ]:
reader = YOLO(WEIGHTS[0])
device = 0 if torch.cuda.is_available() else "cpu"

rows = []
for start in range(0, len(test_images), 64):
    batch = test_images[start:start + 64]
    results = reader.predict(batch, imgsz=320, conf=0.25, device=device, verbose=False)
    for path, r in zip(batch, results):
        chars = list(zip(r.boxes.cls.int().tolist(), r.boxes.xywhn[:, 0].tolist()))
        truth = read_truth(path)
        pred = plate_text(chars)
        rows.append({"image": path.name, "source": path.name.split("__")[0],
                     "truth": truth, "predicted": pred, "correct": truth == pred,
                     "errors": edit_distance(truth, pred), "length": len(truth)})

baseline = pd.DataFrame(rows)
baseline.head()

In [ ]:
def summary(df):
    return pd.Series({"plates": len(df),
                      "plate acc %": round(100 * df["correct"].mean(), 1),
                      "CER %": round(100 * df["errors"].sum() / df["length"].sum(), 2)})


table = baseline.groupby("source").apply(summary)
table.loc["all"] = summary(baseline)
table

## Plate rules: the 5 best valid readings

A Bangladeshi plate has one district, at most one word (Metro / Chatto / Raj), one series letter and six digits.

The reader is run again at 0.05 confidence so every character position keeps its weaker guesses too.
Guesses that overlap are one position. Each position either takes one of its (up to 3) guesses, scored `log(conf)`,
or is skipped as noise, scored `log(1 - best conf)`. A beam search keeps only readings that follow the plate pattern,
and the 5 best go to the redrawing step.

In [ ]:
def reader_guesses(images, conf=0.05):
    out = {}
    for start in range(0, len(images), 64):
        batch = images[start:start + 64]
        results = reader.predict(batch, imgsz=320, conf=conf, device=device, max_det=100, verbose=False)
        for path, r in zip(batch, results):
            out[path.name] = [
                {"cls": c, "conf": p, "x": xywh[0], "box": xyxy}
                for c, p, xywh, xyxy in zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist(),
                                            r.boxes.xywhn.tolist(), r.boxes.xyxyn.tolist())
            ]
    return out


test_guesses = reader_guesses(test_images)
print(sum(len(g) for g in test_guesses.values()), "guesses on", len(test_guesses), "plates")

In [ ]:
import math

LIMIT = {"digit": 6, "district": 1, "letter": 1, "word": 1}
KINDS = list(LIMIT)


def iou(a, b):
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = w * h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


def positions(guesses):
    # overlapping guesses are one character position, best guess first
    groups = []
    for g in sorted(guesses, key=lambda g: -g["conf"]):
        for group in groups:
            if iou(group[0]["box"], g["box"]) > 0.5:
                group.append(g)
                break
        else:
            groups.append([g])
    return sorted(groups, key=lambda group: group[0]["x"])


def top_readings(guesses, k=5, beam=256):
    groups = positions(guesses)
    # a state is (score, counts per kind, chosen guesses)
    states = [(0.0, (0, 0, 0, 0), ())]
    for i, group in enumerate(groups):
        left = len(groups) - i - 1
        options = group[:3]
        new_states = []
        for score, counts, chosen in states:
            # skip this position
            new_states.append((score + math.log(max(1e-4, 1 - options[0]["conf"])), counts, chosen))
            # or take one of its guesses
            for g in options:
                j = KINDS.index(kind[g["cls"]])
                c = list(counts)
                c[j] += 1
                if c[j] <= LIMIT[KINDS[j]]:
                    new_states.append((score + math.log(max(1e-4, g["conf"])), tuple(c), chosen + (g,)))
        # drop states that can no longer reach six digits, a district and a letter
        new_states = [s for s in new_states if (6 - s[1][0]) + (1 - s[1][1]) + (1 - s[1][2]) <= left]
        states = sorted(new_states, key=lambda s: -s[0])[:beam]

    readings = {}
    for score, counts, chosen in states:
        if counts[:3] == (6, 1, 1):
            text = plate_text([(g["cls"], g["x"]) for g in chosen])
            if text not in readings or score > readings[text]["logp"]:
                readings[text] = {"text": text, "logp": score, "chars": [(g["cls"], g["box"]) for g in chosen]}
    return sorted(readings.values(), key=lambda r: -r["logp"])[:k]


example = test_images[0].name
for r in top_readings(test_guesses[example]):
    print(round(r["logp"], 2), r["text"])
print("truth:", read_truth(test_images[0]))

In [ ]:
rules = []
for path in test_images:
    readings = top_readings(test_guesses[path.name])
    truth = read_truth(path)
    pred = readings[0]["text"] if readings else ""
    rules.append({"image": path.name, "source": path.name.split("__")[0], "truth": truth,
                  "predicted": pred, "correct": truth == pred,
                  "errors": edit_distance(truth, pred), "length": len(truth),
                  "in_top5": any(r["text"] == truth for r in readings), "n_readings": len(readings)})
rules = pd.DataFrame(rules)

compare = pd.DataFrame({
    "baseline acc %": baseline.groupby("source")["correct"].mean() * 100,
    "plate rules acc %": rules.groupby("source")["correct"].mean() * 100,
    "truth in top 5 %": rules.groupby("source")["in_top5"].mean() * 100,
})
compare.loc["all"] = [baseline["correct"].mean() * 100, rules["correct"].mean() * 100, rules["in_top5"].mean() * 100]
print("plates with no valid reading:", (rules["n_readings"] == 0).sum())
compare.round(1)

## Glyph atlas

To draw a plate we need one clean image per character class. For each class we cut its boxes out of the training crops,
keep the 40 tallest (the sharpest), and take their pixel-wise median. No network is trained here.

Every crop is first turned grey, its contrast stretched to 0–1, and flipped if needed so the ink is dark on a light plate.

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

PH, PW = 80, 160   # every plate is compared at this size
GLYPH = 48         # size of one atlas image
PAD = 0.08         # margin around a character box, the same when cutting and drawing


def normalise(gray):
    lo, hi = np.percentile(gray, [2, 98])
    g = np.clip((gray - lo) / max(hi - lo, 1e-3), 0, 1).astype(np.float32)
    if np.median(g) < 0.5:   # light ink on a dark plate
        g = 1 - g
    return g


def load_plate(path):
    # returns the normalised crop at full size and at the compare size
    full = normalise(np.asarray(Image.open(path).convert("L"), np.float32) / 255)
    small = Image.fromarray((full * 255).astype(np.uint8)).resize((PW, PH), Image.BILINEAR)
    return full, np.asarray(small, np.float32) / 255


def to_corners(box):
    # YOLO (x centre, y centre, w, h) -> (x0, y0, x1, y1), all relative to the crop
    x, y, w, h = box
    return (x - w / 2, y - h / 2, x + w / 2, y + h / 2)


def padded(box):
    x0, y0, x1, y1 = box
    dx, dy = (x1 - x0) * PAD, (y1 - y0) * PAD
    return (x0 - dx, y0 - dy, x1 + dx, y1 + dy)


def resize(img, w, h):
    return np.asarray(Image.fromarray((img * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR), np.float32) / 255

In [ ]:
from collections import defaultdict

train_images = sorted((DATA / "images" / "train").iterdir())
pool = defaultdict(list)   # class -> [(height in pixels, glyph image)]

for path in train_images:
    full, _ = load_plate(path)
    h, w = full.shape
    for cls, *box in read_labels(path, "train"):
        if len(pool[cls]) >= 400:
            continue
        x0, y0, x1, y1 = padded(to_corners(box))
        x0, x1 = int(max(0, x0 * w)), int(min(w, x1 * w))
        y0, y1 = int(max(0, y0 * h)), int(min(h, y1 * h))
        if x1 - x0 >= 3 and y1 - y0 >= 3:
            pool[cls].append((y1 - y0, resize(full[y0:y1, x0:x1], GLYPH, GLYPH)))

atlas = {}
for cls, items in pool.items():
    items.sort(key=lambda t: -t[0])
    atlas[cls] = np.median(np.stack([g for _, g in items[:40]]), axis=0)

missing = [classes[c]["name"] for c in range(len(classes)) if c not in atlas]
print(len(train_images), "training crops")
print(len(atlas), "of", len(classes), "classes in the atlas, missing:", missing or "none")
np.savez_compressed(WORK / "glyph_atlas.npz", **{str(c): g for c, g in atlas.items()})

In [ ]:
cols = 16
rows = math.ceil(len(atlas) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 0.9, rows))
for ax in axes.flat:
    ax.axis("off")
for ax, cls in zip(axes.flat, sorted(atlas)):
    ax.imshow(atlas[cls], cmap="gray", vmin=0, vmax=1)
    ax.set_title(classes[cls]["name"][:8], fontsize=6)
plt.tight_layout()
plt.savefig(WORK / "atlas.png", dpi=130)
plt.show()

## Drawing a plate

A reading is drawn on a white plate: each character's atlas image is pasted into its box, and a dash is drawn between the
second and third digit. Below, test plates drawn from their **true** labels next to the photo, to check the atlas looks right.

In [ ]:
def draw_plate(chars):
    # chars: list of (class id, (x0, y0, x1, y1)) relative to the crop
    plate = np.ones((PH, PW), np.float32)
    for cls, box in chars:
        if cls not in atlas:
            continue
        x0, y0, x1, y1 = padded(box)
        px0, px1 = round(x0 * PW), round(x1 * PW)
        py0, py1 = round(y0 * PH), round(y1 * PH)
        if px1 - px0 < 2 or py1 - py0 < 2:
            continue
        glyph = resize(atlas[cls], px1 - px0, py1 - py0)
        # clip to the plate edges
        cx0, cy0, cx1, cy1 = max(px0, 0), max(py0, 0), min(px1, PW), min(py1, PH)
        if cx1 > cx0 and cy1 > cy0:
            part = glyph[cy0 - py0:cy1 - py0, cx0 - px0:cx1 - px0]
            plate[cy0:cy1, cx0:cx1] = np.minimum(plate[cy0:cy1, cx0:cx1], part)

    digits = sorted([box for cls, box in chars if kind[cls] == "digit"])
    if len(digits) == 6 and digits[2][0] > digits[1][2]:
        left, right = digits[1][2], digits[2][0]
        yc = np.mean([(b[1] + b[3]) / 2 for b in digits])
        dh = np.mean([b[3] - b[1] for b in digits])
        x0, x1 = round((left + 0.2 * (right - left)) * PW), round((right - 0.2 * (right - left)) * PW)
        y0 = round((yc - 0.06 * dh) * PH)
        plate[max(y0, 0):max(y0 + 1, round((yc + 0.06 * dh) * PH)), max(x0, 0):x1] = 0.0
    return plate


fig, axes = plt.subplots(6, 2, figsize=(6, 9))
for row, path in zip(axes, test_images[::len(test_images) // 6]):
    _, photo = load_plate(path)
    truth_chars = [(cls, to_corners(box)) for cls, *box in read_labels(path)]
    row[0].imshow(photo, cmap="gray", vmin=0, vmax=1)
    row[0].set_title("photo", fontsize=8)
    row[1].imshow(draw_plate(truth_chars), cmap="gray", vmin=0, vmax=1)
    row[1].set_title("drawn", fontsize=8)
    for ax in row:
        ax.axis("off")
plt.tight_layout()
plt.savefig(WORK / "drawn_vs_photo.png", dpi=130)
plt.show()